# TP2 - Parte 1: dataset y preprocesamiento

Notebook de documentación y tratamiento del dataset **70 Dog Breeds Image Dataset**: análisis
exploratorio, control de calidad, conjunto independiente de evaluación, preprocesamiento,
data augmentation y splits (ver `ENUNCIADO.md`, sección 3).

Tiene que poder leerse como un informe: celdas Markdown que explican qué se hace y por qué,
código y salidas (gráficos y tablas) **ejecutadas y guardadas**. Una notebook sin salidas
visibles se considera no entregada.

Lo que produce esta notebook lo reutilizan `2-train.ipynb` y `train_classifier`: los splits
finales (CSV/JSON con rutas y etiquetas), el conjunto independiente y el preprocesamiento
encapsulado en un módulo de `src/lib/`.

## Equipo
- Alumno 1 : Máximo Alva
- Alumno 2 : Zahir Jacob

## 1. Configuración del entorno

En Colab no hay PostgreSQL, por lo que se desactiva pgvector y se usa la base vectorial JSON.
La configuración se define por variables de entorno **antes** de importar `lib`.

In [25]:
import os
import sys
from pathlib import Path
import torch
import pandas as pd
from PIL import Image

ROOT = Path.cwd()

sys.path.insert(0, str(ROOT / "src"))
os.environ["DATASET_PATH"] = str(ROOT / "data" / "dataset")
os.environ["USE_PGVECTOR"] = "false"

from lib.config import settings
from lib.dataset import normalize_breed_folders

DATASET = Path(settings.dataset_path)
SPLITS = ("train", "valid", "test")
SEED = 42

print("dataset_path:", settings.dataset_path)
print("GPU disponible:", torch.cuda.is_available())

dataset_path: data\dataset
GPU disponible: False


## 2. Descargar el dataset

In [ ]:
!python scripts/download_dataset.py
!ls data/dataset

## 3. Descripción del dataset

TO-DO:
- Cantidad de razas, imágenes por raza en cada split, formato, resolución y licencia de uso.
- Cómo están encuadradas las imágenes, comparado con los recortes de YOLO que va a recibir el
  clasificador en la Etapa 3.
- Limitaciones del dataset (razas parecidas, sesgos, etc.) y cómo pueden afectar los resultados.

### 3.1. Inventario de archivos

Recorremos el dataset para armar un `DataFrame` con **una fila por imagen**: split, raza, ruta (relativa a `data/dataset`), extensión, tamaño en disco y metadata de la imagen (ancho, alto, modo de color y formato).

In [ ]:
def build_inventory(dataset_dir: Path, splits: tuple[str, ...]) -> pd.DataFrame:
    """
    Arma un inventario de las imágenes del dataset, con metadata como raza, ruta,
    extensión, tamaño y propiedades de la imagen.

    Args:
        dataset_dir (Path): Carpeta raíz del dataset.
        splits (tuple[str, ...]): Splits a recorrer (por ejemplo, ("train", "valid", "test")).

    Returns:
        pd.DataFrame: Inventario con una fila por imagen y su metadata.
    """
    rows = []
    for split in splits:
        for breed_dir in sorted(p for p in (dataset_dir / split).iterdir() if p.is_dir()):
            for f in sorted(p for p in breed_dir.iterdir() if p.is_file()):
                row = {
                    "split": split,
                    "breed": breed_dir.name,
                    "path": f.relative_to(dataset_dir).as_posix(),
                    "ext": f.suffix.lower(),
                    "size_kb": f.stat().st_size / 1024,
                }
                try:
                    with Image.open(f) as im:  # lee solo el header, no decodifica
                        row.update(width=im.width, height=im.height,
                                   mode=im.mode, format=im.format, opens=True)
                except Exception:
                    row.update(width=None, height=None, mode=None, format=None, opens=False)
                rows.append(row)
    return pd.DataFrame(rows)

In [27]:
df = build_inventory(DATASET, SPLITS)
print(f"Total de archivos: {len(df)}")
df.head()

Total de archivos: 9346


,split,breed,path,ext,size_kb,width,height,mode,format,opens
0,train,Afghan,train/Afghan/001.jpg,.jpg,19.914062,224,224,RGB,JPEG,True
1,train,Afghan,train/Afghan/002.jpg,.jpg,20.272461,224,224,RGB,JPEG,True
2,train,Afghan,train/Afghan/003.jpg,.jpg,21.757812,224,224,RGB,JPEG,True
3,train,Afghan,train/Afghan/004.jpg,.jpg,16.305664,224,224,RGB,JPEG,True
4,train,Afghan,train/Afghan/005.jpg,.jpg,24.864258,224,224,RGB,JPEG,True


### 3.2. Consistencia de etiquetas

Antes de contar razas se verifica que los tres splits tengan exactamente el mismo conjunto de nombres. La etiqueta de cada imagen es el nombre de su carpeta.

In [29]:
def report_breeds(df: pd.DataFrame) -> None:
    """
    Imprime cuántas razas hay en total y en cada split, y cuáles faltan en cada uno.
    
    Args:
        df (pd.DataFrame): Inventario de imágenes con columna "breed" y "split".
    """
    breeds_per_split = df.groupby("split")["breed"].unique().apply(set)
    all_names = set().union(*breeds_per_split)
    print(f"Razas totales: {len(all_names)}")
    for split in SPLITS:
        missing = all_names - breeds_per_split[split]
        print(f"Razas {split:7s}: {len(breeds_per_split[split])}"
                + (f" | Raza/s faltante/s: {sorted(missing)}" if missing else ""))

In [30]:
report_breeds(df)

Razas totales: 71
Razas train  : 70 | Raza/s faltante/s: ['American  Spaniel']
Razas valid  : 70 | Raza/s faltante/s: ['American Spaniel']
Razas test   : 70 | Raza/s faltante/s: ['American  Spaniel']


En el dataset original hay 71 razas en total y en cada split hay 70, es decir, a todos les falta 1. La raza *American Spaniel* figura como
`American Spaniel` en `train` y `test`, y como `American  Spaniel` (dos espacios) en `valid`. Sin corregirlo, las imágenes de esa raza en `valid` tendrían una etiqueta que no existe en `train` por lo que el modelo nunca podría acertarlas y la accuracy de validación quedaría sesgada.

Tomamos la decisión de renombrar las carpetas en disco. `scripts/build_index.py` y
`torchvision.datasets.ImageFolder` toman la raza directamente del nombre de la carpeta, así que corregirlo
solo en el `DataFrame` obligaría a parchear cada lugar que lea el dataset. Renombrar la carpeta deja un
único punto de corrección.

Para que sea reproducible, la corrección está en `src/lib/dataset.py` (`normalize_breed_folders`), esta función colapsa los espacios repetidos de los nombres de carpeta y es idempotente. Se llama acá y también en `scripts/download_dataset.py`, así cualquier nueva descarga queda automáticamente corregida.

In [ ]:
renamed = normalize_breed_folders(DATASET, SPLITS)
if not renamed:
    df = build_inventory(DATASET, SPLITS) # se rehace el inventario con los nombres corregidos
else:
    print("Las carpetas ya estaban normalizadas, no se renombró nada.")

INFO:lib.dataset:valid: 'American  Spaniel' -> 'American Spaniel'


valid: 'American  Spaniel' -> 'American Spaniel'


In [32]:
report_breeds(df)

Razas totales: 70
Razas train  : 70
Razas valid  : 70
Razas test   : 70


### 3.3. Cantidad de razas e imágenes por raza en cada split

In [ ]:
counts = df.pivot_table(index="breed", columns="split", values="path", aggfunc="count", fill_value=0)[list(SPLITS)]

summary = pd.DataFrame({
    "razas": (counts > 0).sum(),
    "imágenes": counts.sum(),
    "% del total": (counts.sum() / counts.values.sum() * 100).round(1),
    "media/raza": counts.mean().round(1),
    "mediana/raza": counts.median(),
    "mín/raza": counts.min(),
    "máx/raza": counts.max(),
}).T
summary["total"] = [counts.index.size, counts.values.sum(), 100.0, None, None, None, None]
summary

split,train,valid,test,total
razas,70.0,70.0,70.0,70.0
imágenes,7946.0,700.0,700.0,9346.0
% del total,85.0,7.5,7.5,100.0
media/raza,113.5,10.0,10.0,NaN
mediana/raza,111.5,10.0,10.0,NaN
mín/raza,65.0,10.0,10.0,NaN
máx/raza,198.0,10.0,10.0,NaN


El dataset tiene **9346 imágenes de 70 razas**, divididas en `train` (7946, 85 %), `valid` (700, 7,5 %) y `test` (700, 7,5 %). Las 70 razas están presentes en los tres splits.

`train` está **desbalanceado**, tiene entre 65 y 198 imágenes por raza, es decir, la raza más representada tiene 3 veces más imágenes que la menos representada. La media (113,5) y la mediana (111,5) son casi iguales, lo que indica una distribución simétrica, sin muchos outliers. Apliaremos el análisis en el EDA.

`valid` y `test` están perfectamente balanceados, todas las razas presentan 10 imágenes.

### 3.4. Formato, resolución y canales

In [41]:
res = df["width"].astype("Int64").astype(str) + "x" + df["height"].astype("Int64").astype(str)

for title, series in [("Formatos", df["format"]), ("Extensiones", df["ext"]),
                      ("Modos de color", df["mode"]), ("Resoluciones", res)]:
    print(title)
    print(series.value_counts(dropna=False).rename_axis(None).to_string(), end="\n\n")

print(f"Archivos que no abren: {(~df['opens']).sum()}")
print(f"Tamaño en disco: {df['size_kb'].sum() / 1024:.1f} MB "
      f"(media {df['size_kb'].mean():.1f} KB por imagen)")

Formatos
JPEG    9346

Extensiones
.jpg    9346

Modos de color
RGB    9346

Resoluciones
224x224    9346

Archivos que no abren: 0
Tamaño en disco: 215.0 MB (media 23.6 KB por imagen)


Las 9346 imágenes son **JPEG, RGB y de 224×224 píxeles**. Todas se pueden abrir (la verificación
completa de archivos corruptos se realiza en la sección 5). La resolución fija indica que las imágenes ya fueron recortadas y/o redimensionadas a 224×224. No podemos conocer la relación de aspecto original.

### 3.5. Licencia de uso

El dataset se distribuye bajo **CC0 1.0 Universal** (public domain): permite copiar, modificar y distribuir las imágenes, incluso con fines comerciales, sin pedir permiso ni exigir atribución. El uso académico de este trabajo está cubierto.

### 3.6. Encuadre de las imágenes vs. recortes de YOLO

### 3.7. Limitaciones del dataset

## 4. Análisis exploratorio (EDA)

TO-DO:
- Distribución de clases: imágenes por raza en cada split (gráfico y estadísticas). Desbalance.
- Resoluciones, relación de aspecto, canales, formatos y archivos corruptos.
- Grilla de ejemplos por raza, incluyendo pares de razas fáciles de confundir.
- Variaciones: iluminación, pose, fondo, oclusiones y cantidad de perros por imagen.

In [ ]:
## TO-DO: análisis exploratorio del dataset.

## 5. Control de calidad y filtrado

TO-DO: reglas explícitas y reproducibles (archivos corruptos, baja resolución, imágenes borrosas o
con exposición extrema, duplicados y casi-duplicados dentro y entre splits, imágenes mal etiquetadas).
Reportar cuántas imágenes descarta cada regla y mostrar ejemplos de descartes.

In [ ]:
## TO-DO: reglas de filtrado, conteo de descartes por regla y ejemplos.

## 6. Conjunto independiente de evaluación

TO-DO: imágenes que no pertenezcan al dataset (internet o fotos propias), con las mismas razas que
las carpetas del dataset. Incluir imágenes con un perro (Etapas 1 y 2), escenas con varios perros u
otros objetos e imágenes sin perros (Etapa 3). Documentar fuente y licencia; no subirlas al repositorio.

In [ ]:
## TO-DO: construcción y resumen del conjunto independiente (por ejemplo, en data/dataset/independent/<raza>/).

## 7. Preprocesamiento

TO-DO: definir y justificar el resize, la normalización y el orden de canales (la app entrega BGR).

⚠️ El mismo preprocesamiento debe usarse en las notebooks y en los servicios de la app:
encapsulalo en un módulo de `src/lib/` e importalo desde ambos lados.

In [ ]:
## TO-DO: transformaciones de preprocesamiento (importadas desde el módulo compartido).

## 8. Data augmentation

TO-DO: horizontal flip, rotación, blur, variaciones de brillo y contraste y ruido, con sus rangos
justificados. Mostrar una grilla de ejemplos aumentados de una misma imagen. Solo sobre `train`.

In [ ]:
## TO-DO: pipeline de data augmentation con ejemplos visuales.

## 9. Splits

TO-DO: definir train / validation / test (los del dataset o redefinidos, justificando), verificar
que no haya fuga de datos entre splits, fijar semillas y guardar los splits finales (CSV/JSON con
rutas y etiquetas) para `2-train.ipynb`.

In [ ]:
## TO-DO: splits finales, verificación de fuga de datos y exportación.

## 10. Resumen final

TO-DO: tabla con la cantidad de razas e imágenes por split, incluido el conjunto independiente.

In [ ]:
## TO-DO: tabla resumen.

## 11. Licencias y uso de imágenes

TO-DO: licencia del dataset, fuente y licencia de las imágenes del conjunto independiente y criterio
aplicado a imágenes con personas identificables.